# One Look · 02 · Données sur Colab

Ce notebook prépare une session Google Colab : il clone le dépôt, installe l'environnement avec **uv**, télécharge le paquet d'entraînement depuis Hugging Face et le vérifie. Les entraînements des notebooks suivants partent de cet état.

**Avant de lancer :**

1. *Exécution → Modifier le type d'exécution → GPU (T4)*.
2. Dans le panneau 🔑 *Secrets* de Colab, ajouter `HF_TOKEN` (une clé Hugging Face en lecture) et activer l'accès pour ce notebook.

Le code vit dans le dépôt (`scripts/`, `src/onelook/`) : ce notebook ne fait que le lancer.

In [ ]:
# Dépôt du paquet sur Hugging Face (privé)
HF_REPO = "Elfried9684/onelook-alert-data"
REPO_URL = "https://github.com/elfried96/onelook-alert_Elf.git"
WORKDIR = "/content/onelook-alert_Elf"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : changer le type d'exécution"

## 1. Code et environnement (uv)

In [ ]:
import os
if not os.path.exists(WORKDIR):
    !git clone -q {REPO_URL} {WORKDIR}
else:
    !git -C {WORKDIR} pull -q
%cd {WORKDIR}
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!uv sync --frozen --group train
!git log --oneline -1

## 2. Paquet d'entraînement (environ 7,5 Go)

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!uv run hf download {HF_REPO} --repo-type dataset --local-dir data/pack --quiet
!du -sh data/pack

## 3. Vérification

In [ ]:
!uv run python scripts/check_pack.py

## Suite

Le paquet est sur le disque de la session (`data/pack/`), effacé à la fermeture de Colab : la cellule 2 le retélécharge à chaque nouvelle session, en quelques minutes.